## Jedi Robes, Inc

Production planning problems are like the basic Jedi Robes, Inc. but with more items. Suppose Jedi Robes, Inc., expands to producing 5 clothing items by using two different production processes: weaving and infusing. Each clothing item requires a number of hours to produce and contributes a fixed number of credits to JRI’s profit. Each unit of each product requires 20 hours of labor for final assembly. The production factory has 3 weaving machines and 2 infusion tanks. The facility runs 6 days a week with 2 shifts each day (8 hours/shift). Eight employees work in assembly, each working one shift/day. Figure out how many items to produce each week.


In [2]:
using JuMP, Gurobi

products = 1:5
profit = Dict(zip(products,[50 65 35 0 50]))
weave = Dict(zip(products, [12 20 0 25 15]))
infuse = Dict(zip(products, [10 8 16 0 0]))
assemble = Dict(zip(products, [20 20 20 20 20]))

m = Model(solver=GurobiSolver(OutputFlag=0))

@variable(m, x[products] >= 0)

@objective(m, Max, sum(x[i] * profit[i] for i in products))

@constraint(m, sum(x[i]*weave[i] for i in products) <= 3*6*16)
@constraint(m, sum(x[i]*infuse[i] for i in products) <= 2*6*16)
@constraint(m, sum(x[i]*assemble[i] for i in products) <= 4*16*6)

solve(m)

println(getobjectivevalue(m))
println(getvalue(x))

Academic license - for non-commercial use only
1104.0
x: 1 dimensions:
[1] = 0.0
[2] = 14.400000000000002
[3] = 4.799999999999999
[4] = 0.0
[5] = 0.0


## Logical constraints

Add the requirement that we can produce no more than 2 products.

In [7]:
m = Model(solver=GurobiSolver(OutputFlag=0))

@variable(m, x[products] >= 0)
@variable(m, z[products], Bin)

@objective(m, Max, sum(x[i] * profit[i] for i in products))

@constraint(m, sum(x[i]*weave[i] for i in products) <= 3*6*16)
@constraint(m, sum(x[i]*infuse[i] for i in products) <= 2*6*16)
@constraint(m, sum(x[i]*assemble[i] for i in products) <= 4*16*6)

# value for big M is an upper bound on x
# e.g., we can use the RHS of assembly constraint / assembly hours (20)
M = 4*16*6 / 20

# x > 0 => z = 1
@constraint(m, ub[i in products], x[i] <= z[i]*M)

# no more than 2 products
@constraint(m, sum(z) <= 2)

solve(m)

println(getobjectivevalue(m))
println(getvalue(x))

Academic license - for non-commercial use only
1104.0
x: 1 dimensions:
[1] = 0.0
[2] = 14.4
[3] = 4.8
[4] = 0.0
[5] = 0.0


### More Logical Constraints

Add the requirement that we can't produce "small quantities." In other words, if we produce a product, we must produce at least 5 units of it.

In [6]:
m = Model(solver=GurobiSolver(OutputFlag=0))

@variable(m, x[products] >= 0)
@variable(m, z[products], Bin)

@objective(m, Max, sum(x[i] * profit[i] for i in products))

@constraint(m, sum(x[i]*weave[i] for i in products) <= 3*6*16)
@constraint(m, sum(x[i]*infuse[i] for i in products) <= 2*6*16)
@constraint(m, sum(x[i]*assemble[i] for i in products) <= 4*16*6)

# value for big M is an upper bound on x
# e.g., we can use the RHS of assembly constraint / assembly hours (20)
M = 4*16*6 / 20

# x > 0 => z = 1
@constraint(m, ub[i in products], x[i] <= z[i]*M)

# no more than 2 products
@constraint(m, sum(z) <= 2)

# produce at least 5 of any product where z=1
@constraint(m, lb[i in products], x[i] >= 5*z[i])

solve(m)

println(getobjectivevalue(m))
println(getvalue(x))

Academic license - for non-commercial use only
1085.0000000000007
x: 1 dimensions:
[1] = 0.0
[2] = 14.00000000000001
[3] = 5.0
[4] = 0.0
[5] = 0.0
